# SGLang — Qwen実行計測
サイトで保存した設定を読み込み、実モデルを実行して結果JSONをダウンロードします。

「ランタイム → ランタイムのタイプを変更」でCUDA GPUを選んでください。無料T4での動作は未保証です。余裕のあるGPUメモリ・通常RAM・ディスクが必要です。

この開発環境ではGPU・有効キーでの実行は未検証です。失敗時はエラーで停止し、模擬結果を生成しません。依存関係は公式開発版を含み、環境によって調整が必要です。

Colabは対話的な実験用です。このノートブックは公開URL・トンネルを作りません。実行後はランタイムを停止してください。
[公式インストール手順](https://github.com/sgl-project/sglang/blob/main/docs/docs/sglang-diffusion/installation.mdx)。1GPU・基本設定で実行します。vLLM-Omniとは新しいランタイムに分けてください。モデル読込を含む時間なので、純粋な生成速度の比較ではありません。


## 1. インストール
パッケージを更新した後にランタイム再起動を求められた場合は、再起動し、次のセルから進んでください。

In [ ]:
%pip install uv requests pillow
import subprocess
from pathlib import Path
if not Path('/content/sglang/.git').exists():
    subprocess.run(['git','clone','https://github.com/sgl-project/sglang.git','/content/sglang'],check=True)
subprocess.run(['uv','venv','--python','3.12','--seed','/content/atlas-sglang'],check=True)
subprocess.run(['uv','pip','install','--python','/content/atlas-sglang/bin/python','--prerelease=allow','-e','/content/sglang/python[diffusion]'],check=True)


## 2. 設定を読み込む
サイトから保存した設定JSONを1つ選びます。参照画像は後の実行セルで選びます。

In [ ]:
"""Shared helpers embedded into the Colab notebooks. No public server or tunnel."""
import base64
import datetime
import hashlib
import io
import json
import math
import os
from pathlib import Path
import platform
import subprocess
import sys
import time

KINDS = ('comfyui', 'inference', 'prompt-rewrite', 'evals')

def validate_config(c, expected):
    if c.get('schema') != 'atlas-colab-config-v1' or c.get('kind') != expected:
        raise ValueError('別のデモの設定ファイルです。対象サイトから保存し直してください。')
    if not isinstance(c.get('prompt'), str) or not 1 <= len(c['prompt'].strip()) <= 4500:
        raise ValueError('プロンプトを確認してください。')
    for key, minimum, maximum in [('seed', 0, 2147483647), ('steps', 1, 50), ('repetitions', 1, 20)]:
        if type(c.get(key)) is not int or not minimum <= c[key] <= maximum:
            raise ValueError('設定値が不正です: ' + key)
    if c.get('size') not in (1024, 2048) or c.get('mode') not in ('generate', 'edit'):
        raise ValueError('サイズ・モードが不正です。')
    if not isinstance(c.get('transparent'), bool):
        raise ValueError('透過設定が不正です。')
    return c

def load_config(expected):
    from google.colab import files
    print('サイトから保存した atlas-' + expected + '-config.json を選択してください。')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('設定JSONは1件だけ選択してください。')
    raw = next(iter(uploaded.values()))
    if len(raw) > 20000:
        raise ValueError('設定ファイルが大きすぎます。')
    return validate_config(json.loads(raw), expected)

def gpu_info(required=True):
    try:
        result = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        if required:
            raise RuntimeError('CUDA GPUランタイムを選択してください。')
        result = 'CPU (API evaluation)'
    return {'gpu': result, 'python': platform.python_version()}

def effective_prompt(c):
    return ('This is an RGBA image with transparency. ' + c['prompt'] + ' The image has alpha channel and the background is transparent.') if c['transparent'] else c['prompt']

def upload_images(maximum=10):
    from google.colab import files
    from PIL import Image
    uploaded = files.upload()
    if not 1 <= len(uploaded) <= maximum:
        raise ValueError(f'画像は1〜{maximum}枚です。')
    images = []
    total = 0
    for data in uploaded.values():
        total += len(data)
        if len(data) > 4 * 1024**2 or total > 20 * 1024**2:
            raise ValueError('画像は1枚4MB、合計20MBまでです。')
        img = Image.open(io.BytesIO(data))
        if img.format not in ('PNG', 'JPEG', 'WEBP') or img.width * img.height > 20_000_000:
            raise ValueError('PNG/JPEG/WebP、2000万画素以下を選択してください。')
        img.load()
        images.append(img)
    return images

def image_data(image):
    output = io.BytesIO()
    image.save(output, format='PNG')
    if len(output.getvalue()) > 16 * 1024**2:
        raise ValueError('結果が16MBを超えました。1024pxで再実行してください。')
    return 'data:image/png;base64,' + base64.b64encode(output.getvalue()).decode('ascii')

def report(c, engine, model, elapsed, env, **extra):
    return {'schema': 'atlas-colab-result-v1', 'kind': c['kind'],
            'created_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'engine': engine, 'model': model, 'elapsed_seconds': elapsed,
            'environment': env, 'config': c, **extra}

def save_report(result):
    from google.colab import files
    path = Path('/content/atlas-' + result['kind'] + '-' + result['engine'] + '-result.json')
    path.write_text(json.dumps(result, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    files.download(str(path))
    print('結果JSONをサイトに読み込んでください。APIキーは結果に含めません。')

config=load_config('inference')
print(json.dumps(config,ensure_ascii=False,indent=2))
print(gpu_info(True))


## 3. 実モデルを実行
初回はモデルのダウンロードに時間がかかります。表示される画像・スコアは実行したモデルの結果です。

In [ ]:
from PIL import Image
import uuid

def run_inference(c, engine):
    env = gpu_info()
    if c['mode'] != 'generate':
        raise ValueError('推論比較はテキストからの生成に限定しています。')
    out = Path('/content/atlas-inference-' + uuid.uuid4().hex)
    out.mkdir()
    if engine == 'vllm-omni':
        repo = Path('/content/vllm-omni')
        command = ['/content/atlas-vllm/bin/python', str(repo / 'examples/offline_inference/text_to_image/text_to_image.py'),
                   '--model','Qwen/Qwen-Image-2.1','--prompt', effective_prompt(c),
                   '--width',str(c['size']),'--height',str(c['size']),
                   '--num-inference-steps',str(c['steps']),'--cfg-scale','1.0',
                   '--seed',str(c['seed']),'--output',str(out / 'image.png')]
        env['revision'] = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
    elif engine == 'sglang':
        repo = Path('/content/sglang')
        command = ['/content/atlas-sglang/bin/sglang','generate','--model-path','Qwen/Qwen-Image-2.1',
                   '--prompt',effective_prompt(c),'--height',str(c['size']),'--width',str(c['size']),
                   '--num-inference-steps',str(c['steps']),'--guidance-scale','1',
                   '--seed',str(c['seed']),'--save-output','--output-path',str(out),'--output-file-name','image.png']
        env['revision'] = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
    else:
        raise ValueError('Unknown engine')
    # A first run populates the model cache; it is deliberately not measured.
    print('初回のモデル取得・動作確認。時間は比較対象にしません。')
    subprocess.run(command, check=True)
    first = list(out.glob('*.png'))
    if not first: raise RuntimeError('PNGが出力されませんでした。CLIログを確認してください。')
    before = {str(p):p.stat().st_mtime_ns for p in first}
    print('2回目を計測：プロセス起動・モデル読込・生成・保存を含みます。')
    start = time.perf_counter()
    subprocess.run(command, check=True)
    elapsed = time.perf_counter() - start
    candidates = [p for p in out.glob('*.png') if str(p) not in before or p.stat().st_mtime_ns != before[str(p)]]
    if not candidates: raise RuntimeError('2回目の出力が見つかりません。古い画像は結果に使用しません。')
    path = max(candidates, key=lambda p:p.stat().st_mtime_ns)
    image = Image.open(path); image.load(); display(image)
    return report(c, engine, 'Qwen/Qwen-Image-2.1', elapsed, env, image=image_data(image), timing_scope='process_including_model_load', warmup_runs=1)

result=run_inference(config,'sglang')


## 4. 結果を保存してサイトへ戻る
結果JSONには入力文や生成物が含まれます。対象のデモで「結果JSONを読み込む」を選びます。サイトの読み込みはブラウザ内だけで処理します。

In [ ]:
save_report(result)
